# Series de tiempo y forecasting

Fases 1 y 2: preparación, auditoría, exploración y ventanas temporales. Los gráficos describen el panel; no se usan para ajustar modelos con datos de prueba. Los datos se descargan y procesan por bloques.

## Preparar el panel

La descarga ocupa cerca de 250 MB y el archivo original cerca de 680 MiB. El lector Parquet usa grupos de filas para evitar cargar el panel largo completo.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

RAW_PATH = ROOT / "data" / "raw" / "LD2011_2014.txt"
PANEL_PATH = ROOT / "data" / "processed" / "hourly.parquet"
CLIENT_IDS_PATH = ROOT / "data" / "processed" / "client_ids.csv"

from src.data.download import download_dataset
from src.data.preprocess import preprocess_dataset
from src.data.panel import load_panel_matrix

if not RAW_PATH.exists():
    download_dataset(RAW_PATH.parent)
if not PANEL_PATH.exists() or not CLIENT_IDS_PATH.exists():
    preprocess_dataset(RAW_PATH, PANEL_PATH, PANEL_PATH.parent, seed=42)

panel = load_panel_matrix(PANEL_PATH, CLIENT_IDS_PATH)
print({
    "clientes": len(panel.client_ids),
    "horas_por_cliente": len(panel.timestamps),
    "desde": panel.timestamps[0],
    "hasta": panel.timestamps[-1],
    "forma_y": panel.values.shape,
})

## Exploración

Se comparan perfiles horarios y semanales, feriados nacionales portugueses, escalas por cliente y el período de Navidad y Año Nuevo. Los promedios por cliente para normalización se calculan solo con entrenamiento.

In [ ]:
from src.data.eda import run_eda

eda_summary = run_eda(PANEL_PATH, CLIENT_IDS_PATH, ROOT / "outputs")
display(eda_summary)
display(pd.DataFrame(eda_summary["window_counts"]))
display(pd.read_csv(ROOT / "outputs" / "metrics" / "portugal_holidays.csv"))

In [ ]:
from IPython.display import Image, display

for name in (
    "hourly_weekly_profiles.png",
    "portugal_holidays.png",
    "client_scales.png",
    "christmas_new_year.png",
):
    print(name)
    display(Image(filename=str(ROOT / "outputs" / "figures" / "eda" / name)))

## Calendario y normalización

Se conserva la hora local sin zona horaria. La codificación cíclica usa 7 columnas para hora, día de semana, mes y feriado; la one-hot usa 44. Se elige cíclica para reducir dimensión y mantener cercanos los valores adyacentes, sin tratar enero y diciembre como categorías lejanas. Las embeddings se reservan para modelos deep que las justifiquen; añaden parámetros y aquí las categorías tienen ciclos conocidos. Las estadísticas globales o por cliente se ajustan solo entre 2012-01-01 y 2014-07-01; la normalización por ventana usa únicamente el historial de entrada.

In [ ]:
from src.data.calendars import calendar_features
from src.data.splits import DEFAULT_SPLITS, temporal_split_origins, daily_backtest_origins
from src.training.scaling import TemporalScaler

cyclic = calendar_features(panel.timestamps, encoding="cyclic")
onehot = calendar_features(panel.timestamps, encoding="onehot")
train_start, train_end = DEFAULT_SPLITS["train"]
scaler = TemporalScaler("per_series").fit(
    panel.values, panel.timestamps, train_start=train_start, train_end=train_end
)
scaled_values = scaler.transform(panel.values)
restored_values = scaler.inverse_transform(scaled_values)
print({
    "features_ciclicas": cyclic.shape[1],
    "features_onehot": onehot.shape[1],
    "error_maximo_al_invertir": float(abs(restored_values - panel.values).max()),
    "fit_desde": scaler.fit_start_,
    "fit_hasta_exclusivo": scaler.fit_end_,
})

## Ventanas L=168 y H=24

Cada origen identifica la primera hora objetivo. El historial termina una hora antes del origen; las covariables futuras se entregan en un tensor separado. Los targets deben quedar enteros dentro de su split. El Dataset de entrenamiento usa stride 1; el backtest de prueba usa 92 orígenes diarios.

In [ ]:
from src.data.windows import MultiSeriesWindowDataset

origins = temporal_split_origins(
    panel.timestamps, lookback=168, horizon=24, stride=1
)
daily_origins = daily_backtest_origins(panel.timestamps)
train_dataset = MultiSeriesWindowDataset(
    scaled_values,
    panel.timestamps,
    panel.client_ids,
    origins["train"][:2],
    lookback=168,
    horizon=24,
    calendar=cyclic,
)
sample = train_dataset[0]
print({
    "X": tuple(sample["X"].shape),
    "y": tuple(sample["y"].shape),
    "calendario_futuro": tuple(sample["future_calendar"].shape),
    "cliente": sample["unique_id"],
    "origen": sample["origin"],
    "primer_objetivo": sample["target_timestamps"][0],
    "ventanas_stride_1_por_split_y_cliente": {
        name: len(indices) for name, indices in origins.items()
    },
    "origenes_diarios_test": len(daily_origins),
})

## Pruebas contra fugas

Estas pruebas detectan targets compartidos o fuera de orden, normalización ajustada con datos posteriores, early stopping sobre test y calendarios futuros desalineados.

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(result.stdout)